# 04 — E2: 20% Node Crash + Recovery (Problem → Solution)

**Core story of the paper**: At cycle 10, 20 of 100 nodes crash.
- **Ring** (no healing): topology holes → convergence collapses
- **Gossip**: degrades but recovers slowly
- **HEAL**: self-elects new hubs within 1–2 cycles → minimal accuracy drop

Recovery metrics: pre-failure acc, max drop, cycles-to-recover

In [ ]:
import subprocess, sys, os

repo_root = os.getcwd()
if not os.path.isdir(os.path.join(repo_root, "src")):
    repo_root = os.path.dirname(repo_root)

assert os.path.isdir(os.path.join(repo_root, "src")), \
    f"src/ not found. cwd={os.getcwd()} — open this notebook from the fl-rockstar repo root"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "datasets", "networkx",
    "numpy", "scipy", "matplotlib", "seaborn", "pandas",
    "tqdm", "pyyaml", "--quiet"])

import copy, random
from functools import partial
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.data import make_noniid_cifar10
from src.model import SimpleCNN
from src.topology import build_elevator_topology, build_ring_topology, build_gossip_topology
from src.simulation import HealClient, HealHub, run_simulation
from src.failures import inject_crash
from src.evaluation import compute_recovery_metrics

SEED = 42
N_NODES = 100
N_HUBS = 5
N_CYCLES = 300
CRASH_CYCLE = 10
CRASH_RATE = 0.2

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")

random.seed(SEED)
torch.manual_seed(SEED)
print(f"device: {device}  |  root: {repo_root}")

In [ ]:
node_data, test_loader = make_noniid_cifar10(N_NODES, shards_per_node=2, seed=SEED)
init_model = SimpleCNN()

In [ ]:
failure_schedule = {
    CRASH_CYCLE: partial(inject_crash, crash_rate=CRASH_RATE, n_hubs=N_HUBS, seed=SEED)
}

results = {}
for cond in ['heal', 'ring', 'gossip']:
    print(f'Running {cond}...')
    if cond == 'heal':
        G, hub_ids = build_elevator_topology(N_NODES, N_HUBS, SEED)
        hubs    = [HealHub(i,    copy.deepcopy(init_model), node_data[i], device=device) for i in hub_ids]
        clients = [HealClient(i, copy.deepcopy(init_model), node_data[i], device=device)
                   for i in range(N_NODES) if i not in hub_ids]
    else:
        G, hub_ids = (build_ring_topology if cond == 'ring' else build_gossip_topology)(N_NODES)
        hubs = []
        clients = [HealClient(i, copy.deepcopy(init_model), node_data[i], device=device) for i in range(N_NODES)]

    hist = run_simulation(cond, N_CYCLES, clients, hubs, hub_ids, G, test_loader,
                          failure_schedule=failure_schedule, eval_every=10)
    results[cond] = hist

print('Done.')

In [ ]:
for cond, hist in results.items():
    m = compute_recovery_metrics(hist, CRASH_CYCLE)
    print(f'\n{cond.upper()} recovery metrics:')
    for k, v in m.items():
        print(f'  {k}: {v}')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

styles = {
    'heal':   ('HEAL (self-healing)',    '#0284C7', '-',  2.5),
    'gossip': ('Gossip',                 '#F59E0B', '--', 1.8),
    'ring':   ('Ring (no healing)',      '#EF4444', ':',  1.8),
}

for cond, (label, color, ls, lw) in styles.items():
    cycles, accs = zip(*results[cond])
    ax.plot(cycles, accs, label=label, color=color, linestyle=ls, linewidth=lw)

ax.axvline(x=CRASH_CYCLE, color='black', linestyle='--', alpha=0.6, linewidth=1.5)
ax.text(CRASH_CYCLE + 2, 0.08, f'Crash @ cycle {CRASH_CYCLE}\n(20% nodes fail)',
        fontsize=9, color='black', alpha=0.8)

ax.set_xlabel('Training Cycle', fontsize=12)
ax.set_ylabel('Average Test Accuracy', fontsize=12)
ax.set_title('E2: 20% Node Crash — Problem (Ring) vs Solution (HEAL)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../results/04_e2_crash_recovery.png', dpi=150)
plt.show()